In [ ]:
!pip install -q langchain==0.2.11 langchain-community==0.2.10 langchain-core==0.2.23 langchain-text-splitters==0.2.2 langchain-huggingface==0.0.3 faiss-cpu pypdf bitsandbytes accelerate transformers sentence-transformers

print("✅ Installation terminée.")
print("⚠️ OBLIGATOIRE : Menu 'Exécution' > 'Redémarrer la session' maintenant.")

In [ ]:
import torch  # Import principal pour le calcul tensoriel et l'utilisation du GPU
import os  # Gestion du système de fichiers (vérification de l'existence du PDF)
import gc  # Garbage Collector : utile pour vider la VRAM manuellement si besoin (bonnes pratiques)
from pypdf import PdfReader  # Librairie légère pour l'extraction de texte depuis des PDF

# --- IMPORTS ---
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig  # Outils Hugging Face pour charger le LLM et gérer la quantification
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Outil pour découper le texte intelligemment en gardant le contexte
from langchain_community.vectorstores import FAISS  # Base de données vectorielle locale (Facebook AI Similarity Search) très rapide
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline  # Connecteurs pour utiliser les modèles HF dans LangChain
from langchain.chains import RetrievalQA  # Chaîne de logique : Recherche (Retrieval) + Réponse (QA)
from langchain.prompts import PromptTemplate  # Outil pour structurer les instructions données au LLM

# --- 1. CONFIGURATION ET LECTURE DU PDF ---

In [ ]:
chemin_pdf = 'Societe-Generale-Pilier-3_T2-2022_FR.pdf'  # Définition du fichier source

if not os.path.exists(chemin_pdf):  # Vérification de sécurité basique
    print(f"❌ ERREUR : Le fichier '{chemin_pdf}' est introuvable.")
else:
    print(f"📂 Lecture du fichier PDF...")
    lecteur = PdfReader(chemin_pdf)  # Chargement du PDF en mémoire
    texte_brut = ''  # Initialisation de la variable qui contiendra tout le texte
    for page in lecteur.pages:  # Boucle sur chaque page du document
        t = page.extract_text()  # Extraction brute du texte (attention: perd souvent la structure des tableaux)
        if t: texte_brut += t  # Accumulation du texte dans une seule longue chaîne

    # DÉCOUPAGE INTELLIGENT
    print("✂️  Découpage optimisé...")
    decoupeur = RecursiveCharacterTextSplitter(  # Stratégie de découpage "Récursive" : essaie de ne pas couper au milieu d'une phrase
        chunk_size=1000,  # Taille de chaque morceau : 1000 caractères (~1 paragraphe, idéal pour le contexte)
        chunk_overlap=250,  # Chevauchement : 250 caractères répétés entre deux morceaux pour ne pas perdre d'info à la coupe
        separators=["\n\n", "\n", ".", " ", ""]  # Ordre de priorité pour la coupe (Paragraphe > Ligne > Phrase > Mot)
    )
    textes = decoupeur.split_text(texte_brut)  # Exécution du découpage
    print(f"   -> {len(textes)} segments.")  # Log pour vérifier si on a bien généré des chunks

# --- 2. EMBEDDINGS (MÉMOIRE) ---

In [ ]:
print("🧠 Création de la mémoire (Modèle Multilingue Puissant)...")
# Ce modèle est bien meilleur que MiniLM pour comprendre le sens en français
embeddings = HuggingFaceEmbeddings(  # Initialisation du modèle d'embedding (Texte -> Vecteurs numériques)
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",  # Choix critique : modèle Multilingue (car PDF en FR) et performant (MPNet)
    model_kwargs={'device': 'cuda'}  # Force l'utilisation du GPU pour l'indexation (beaucoup plus rapide)
)
docsearch = FAISS.from_texts(textes, embeddings)  # Création de l'index vectoriel (La "base de données" de connaissances)

# --- 3. CHARGEMENT DU MODÈLE qwen ---

In [ ]:
print("🤖 Chargement du modèle Qwen 2.5 (7B Instruct)...")

bnb_config = BitsAndBytesConfig(  # Configuration de la quantification (Compression du modèle)
    load_in_4bit=True,  # Active le mode 4-bit (divise la mémoire requise par ~4, permet de tourner sur GPU T4/RTX)
    bnb_4bit_use_double_quant=True,  # Double quantification pour optimiser encore plus l'espace mémoire
    bnb_4bit_quant_type="nf4",  # Type de données "Normal Float 4", optimal pour conserver la précision des poids
    bnb_4bit_compute_dtype=torch.float16  # Les calculs se font en float16 pour la rapidité, même si le stockage est en 4-bit
)

# Qwen 2.5 est le meilleur modèle 7B actuel, et il n'est pas bloqué par une clé
model_id = "Qwen/Qwen2.5-7B-Instruct"  # Choix du LLM : Qwen 2.5 est SOTA (State of the Art) en open-source, excellent en raisonnement

tokenizer = AutoTokenizer.from_pretrained(model_id)  # Chargement du tokenizer (traducteur Texte <-> Nombres pour le modèle)
model = AutoModelForCausalLM.from_pretrained(  # Chargement du modèle neuronal
    model_id,
    quantization_config=bnb_config,  # Application de la config 4-bit définie plus haut
    device_map="auto"  # Répartition automatique sur le GPU/CPU disponible
)

pipe = pipeline(  # Création du pipeline de génération de texte
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=1024,  # Limite de longueur de la réponse (1024 tokens = réponses longues et détaillées possibles)
    temperature=0.1,  # Température très basse (0.1) : Réduit la créativité pour éviter les hallucinations (crucial en finance)
    repetition_penalty=1.1,  # Pénalité pour empêcher le modèle de répéter les mêmes phrases en boucle
    return_full_text=False  # Ne renvoie que la réponse générée, pas le prompt original
)

llm_local = HuggingFacePipeline(pipeline=pipe)  # Wrapper pour rendre le pipeline compatible avec LangChain

# --- 4. PROMPT FRANÇAIS ---

In [ ]:
# Qwen répond bien aux instructions claires
template_qwen = """<|im_start|>system
Tu es un expert financier. Analyse les extraits de documents suivants pour répondre à la question.
Si tu trouves un nom propre ou un chiffre précis, cite-le.
Si l'information n'est pas dans le contexte, dis "Information non trouvée".
Réponds en français.<|im_end|>
<|im_start|>user
CONTEXTE:
{context}

QUESTION:
{question}<|im_end|>
<|im_start|>assistant
"""  # Utilisation du format ChatML (<|im_start|>) spécifique à Qwen pour maximiser la compréhension des instructions

PROMPT = PromptTemplate(  # Création de l'objet Prompt pour LangChain
    template=template_qwen,
    input_variables=["context", "question"]  # Variables qui seront injectées dynamiquement au moment de la question
)

# --- 5. CRÉATION DE LA CHAÎNE ---

In [ ]:
qa_chain = RetrievalQA.from_chain_type(  # Construction de la chaîne RAG finale
        llm=llm_local,  # Le cerveau (Qwen)
        chain_type="stuff",  # Méthode "Stuff" : On bourre tous les chunks trouvés dans un seul prompt (rapide et efficace)
        # k=8 : On donne beaucoup de contexte pour être sûr d'avoir le tableau
        retriever=docsearch.as_retriever(search_kwargs={"k": 8}),  # Récupère les 8 passages les plus pertinents (Top-k) via FAISS
        chain_type_kwargs={"prompt": PROMPT}  # Injection de notre prompt personnalisé (Expert financier)
    )

# --- 6. INTERROGATION (FORMAT MANUEL) ---

In [ ]:
print("\n" + "="*50)
print("      RÉSULTATS DE L'ANALYSE")
print("="*50)

# --- REQUÊTE 1 ---

In [ ]:
requete = "quel était le montant des RWA (Expositions pondérées) à la société générale?"  # Définition de la question utilisateur
print(f"\n❓ Question : {requete}")
# Note : qa_chain.invoke fait la recherche + la génération en une seule fois
reponse = qa_chain.invoke(requete)  # Exécution du pipeline : Recherche Vectorielle -> Prompt -> Génération LLM
print(f"💡 Réponse : {reponse['result'].strip()}")  # Affichage propre du résultat
print("-" * 30)

# --- REQUÊTE 2 ---

In [ ]:
requete = "Quels sont les principaux indicateurs de performance financière mentionnés dans le document ?"
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)  # Réutilisation de la chaîne pour une nouvelle question
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 3 ---

In [ ]:
requete = "Qui est l'auteur du document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 4 ---

In [ ]:
requete = "Quels sont les risques mentionnés dans le document? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)

# --- REQUÊTE 5 ---

In [ ]:
requete = "Quel est le montant des fonds propres à la fin de la période de déclaration? "
print(f"\n❓ Question : {requete}")
reponse = qa_chain.invoke(requete)
print(f"💡 Réponse : {reponse['result'].strip()}")
print("-" * 30)